## Text Summarization

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_classic.schema import SystemMessage, HumanMessage, AIMessage


load_dotenv()

True

In [2]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [5]:
speech = '''Lorem Ipsum is simply dummy text of the printing and typesetting industry. 
Lorem Ipsum has been the industry's standard dummy text ever since 1966, when designers at Letraset and James Mosley, the librarian at St Bride Printing Library in London, took a 1914 Cicero translation and scrambled it to make dummy text for Letraset's Body Type sheets. 
It has survived not only many decades, but also the leap into electronic typesetting, remaining essentially unchanged. 
It was popularised thanks to these sheets and more recently with desktop publishing software like Aldus PageMaker and Microsoft Word including versions of Lorem Ipsum.
Contrary to popular belief, Lorem Ipsum is not simply random text. It has roots in a piece of classical Latin literature from 45 BC, making it over 2000 years old. 
Richard McClintock, a Latin professor at Hampden-Sydney College in Virginia, looked up one of the more obscure Latin words, consectetur, from a Lorem Ipsum passage, and going through the cites of the word in classical literature, discovered the undoubtable source. 
Lorem Ipsum comes from sections 1.10.32 and 1.10.33 of "de Finibus Bonorum et Malorum" (The Extremes of Good and Evil) by Cicero, written in 45 BC. 
This book is a treatise on the theory of ethics, very popular during the Renaissance. The first line of Lorem Ipsum, "Lorem ipsum dolor sit amet..", comes from a line in section 1.10.32.
'''

chat_message = [
    SystemMessage(content="You are an expert with expertise in summarizing text"),
    HumanMessage(content=f"Please provide a short and concise summary of the following:\n text: {speech}")
]

In [4]:
llm.get_num_tokens(speech)

c:\Users\viren\generative-ai-implementations\langchain\agent\.venv\Lib\site-packages\langchain_core\language_models\base.py:447: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


339

In [ ]:
# First way to summarize the text - provided your text is small
llm.invoke(chat_message)

AIMessage(content='Here is a short and concise summary:\n\nLorem Ipsum, the standard dummy text in the printing industry, originated from a 45 BC Latin literature by Cicero, "de Finibus Bonorum et Malorum". It was popularized in 1966 and has remained unchanged despite the transition to electronic typesetting, with its roots in classical Latin literature making it over 2000 years old.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 377, 'total_tokens': 456, 'completion_time': 0.257274809, 'completion_tokens_details': None, 'prompt_time': 0.035894354, 'prompt_tokens_details': None, 'queue_time': 0.057573075, 'total_time': 0.293169163}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f53e7-36d0-7742-b470-90f788692029-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 377

In [9]:
# Prompt Template Text Summarization
from langchain_classic.chains import LLMChain
from langchain_classic import PromptTemplate

generic_template = '''
Write me a summary of the following Speech:
Speech: {speech}
Translate the precise summary to {language}
'''

prompt = PromptTemplate(
    input_variable = ['speech', 'language'],
    template = generic_template
)

prompt

PromptTemplate(input_variables=['language', 'speech'], input_types={}, partial_variables={}, template='\nWrite me a summary of the following Speech:\nSpeech: {speech}\nTranslate the precise summary to {language}\n')

In [12]:
complete_prompt = prompt.format(speech=speech, language="French")
complete_prompt

'\nWrite me a summary of the following Speech:\nSpeech: Lorem Ipsum is simply dummy text of the printing and typesetting industry. \nLorem Ipsum has been the industry\'s standard dummy text ever since 1966, when designers at Letraset and James Mosley, the librarian at St Bride Printing Library in London, took a 1914 Cicero translation and scrambled it to make dummy text for Letraset\'s Body Type sheets. \nIt has survived not only many decades, but also the leap into electronic typesetting, remaining essentially unchanged. \nIt was popularised thanks to these sheets and more recently with desktop publishing software like Aldus PageMaker and Microsoft Word including versions of Lorem Ipsum.\nContrary to popular belief, Lorem Ipsum is not simply random text. It has roots in a piece of classical Latin literature from 45 BC, making it over 2000 years old. \nRichard McClintock, a Latin professor at Hampden-Sydney College in Virginia, looked up one of the more obscure Latin words, consectetur

In [13]:
llm.get_num_tokens(complete_prompt)

362

In [16]:
llm_chain = LLMChain(llm=llm, prompt=prompt)
summary = llm_chain.run({'speech': speech, 'language': 'Hindi'})

In [17]:
summary

'**English Summary:**\nThe speech discusses the origin and history of Lorem Ipsum, a standard dummy text used in the printing and typesetting industry. It was created in 1966 by designers at Letraset, who scrambled a 1914 translation of Cicero\'s work to make dummy text. Despite being over 2000 years old, with roots in classical Latin literature, Lorem Ipsum has remained largely unchanged and has been popularized through various desktop publishing software. Its origins can be traced back to sections 1.10.32 and 1.10.33 of Cicero\'s book "de Finibus Bonorum et Malorum" (The Extremes of Good and Evil), written in 45 BC.\n\n**हिंदी अनुवाद:**\nभाषण लोरेम इप्सम की उत्पत्ति और इतिहास पर चर्चा करता है, जो मुद्रण और टाइपसेटिंग उद्योग में उपयोग किया जाने वाला एक मानक डमी पाठ है। इसे 1966 में लेट्रासेट के डिजाइनरों द्वारा बनाया गया था, जिन्होंने एक 1914 अनुवाद को सीसेरो के काम में बिखेर दिया था। 2000 से अधिक वर्षों से पुराने होने के बावजूद, जिसकी जड़ें शास्त्रीय लैटिन साहित्य में हैं, लोरेम इप्स

### StuffDocumentChain Text Summarization

In [19]:
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('apjspeech.pdf')
docs = loader.load_and_split()
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [20]:
template = """
Write me a concise and short summary of the following speech,
Speech: {text}
"""

prompt = PromptTemplate(
    input_variables=['text'],
    template=template
)

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm=llm, chain_type='stuff', prompt=prompt, verbose=True)
output_summary = chain.run(docs)
output_summary



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Write me a concise and short summary of the following speech,
Speech: A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online

"Here is a concise and short summary of the speech:\n\nIn his departing speech as President of India, A.P.J. Abdul Kalam reflects on his 5-year tenure and highlights 10 key messages for the country's development. He emphasizes the importance of accelerating development, empowering villages, and mobilizing rural core competence. He also stresses the need to defeat problems, overcome disasters, and connect with each other for societal transformation. Kalam expresses his pride in India's defense forces and encourages the youth to work towards a developed India by 2020. He concludes by outlining his vision for a developed India, where the rural-urban divide is reduced, and education, healthcare, and governance are accessible to all. He thanks the citizens for their love and support and reiterates his commitment to making India a developed nation before 2020."

### Map Reduce to Summarize Large Documents

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

final_documents = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100).split_documents(docs)
final_documents

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [23]:
chunk_prompt = """
Please summarize the below speech:
Speech: '{text}'
Summary:
"""

map_prompt_template = PromptTemplate(
    input_variables = ['text'], 
    template=chunk_prompt
)

In [24]:
final_prompt = """
Provide the final summary of the entire speech with these important points.
Add a Motivational Title, Start the precise summary with an introduction and 
provide the summary in number points for the speech.
Speech: {text}
"""

final_prompt_template = PromptTemplate( 
    input_variables = ['text'], 
    template=final_prompt
)

In [27]:
summary_chain = load_summarize_chain(
    llm=llm, 
    chain_type='map_reduce', 
    map_prompt=map_prompt_template, 
    combine_prompt=final_prompt_template,
    verbose=True)

output_summary = summary_chain.run(final_documents)
output_summary



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Please summarize the below speech:
Speech: 'A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have 

'**Motivational Title: "Empowering a Brighter Future: A Call to Action for a Developed India"**\n\nIntroduction:\nAs the former President of India, A P J Abdul Kalam, concludes his five-year tenure at Rashtrapati Bhavan, he delivers a departing speech that embodies his vision for a prosperous and developed India. The speech is a reflection on his experiences and interactions with various sections of society, including politicians, scientists, academics, and youth. It emphasizes the importance of empowering India\'s youth and villages to achieve a brighter future.\n\nHere is a summary of the speech in key points:\n\n1. **Accelerating Development**: The need to accelerate development and fulfill the aspirations of the youth, with a focus on empowering villages and mobilizing rural core competence.\n2. **Empowering Villages**: The importance of empowering villages through initiatives like PURA (Providing Urban Amenities in Rural Areas) to enable them to access markets and develop economic

### Refine Chain For Summarization

In [28]:
chain = load_summarize_chain(
    llm=llm, 
    chain_type='refine', 
    verbose=True)

output_summary = chain.run(final_documents)
output_summary



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have man

"The provided context does not significantly add new information to the original summary, but rather serves as a conclusion to A.P.J. Abdul Kalam's speech. However, it does reiterate his mission to bring connectivity among the people of India and to instill self-confidence in achieving the goal of a developed nation by 2020. \n\nGiven this, the original summary remains largely sufficient, but a minor refinement can be made to include the aspect of his mission for connectivity and self-confidence among the people. Here is a refined version:\n\nIn his departing speech, A.P.J. Abdul Kalam reflects on his 5-year tenure as President, expressing gratitude for the opportunity to work with people from various walks of life. He highlights key areas crucial for India's growth and transformation, including accelerating development, empowering villages, and mobilizing the 540 million youth below the age of 25 through value-based education and leadership, with a goal of becoming a developed nation 